#Install libraries

In [ ]:
%%capture
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

#Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


#Load Policy + Reference from the v5 SFT adapter

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # fragmentation: optimizer-state alloc OOM'd post-step-1
from unsloth import FastLanguageModel
ADAPTER = "/content/drive/MyDrive/re_lora_v5"  # v5 SFT adapter from the SFT notebook's Save cell
# fallback: "/content/drive/MyDrive/re_ckpt_v5/checkpoint-68" (epoch-1) or checkpoint-136 (final)
# Two 4-bit copies, same as run A (proven on this T4). ref_model=None merged-model variant
# produced loss=0.000 at step 1 (broken ref pass) - do not use it.
model, tokenizer = FastLanguageModel.from_pretrained(ADAPTER, max_seq_length=8192, load_in_4bit=True)
ref_model, _ = FastLanguageModel.from_pretrained(ADAPTER, max_seq_length=8192, load_in_4bit=True)
# policy trains the v5 adapter weights further; ref stays frozen = the SFT policy we anchor to

#Data

In [ ]:
import json
from datasets import Dataset
# dpo_v5_runb.jsonl = run-A 51 (39 on-policy re-v5 rejections + 12 synth early-quit)
# + dpo_v5_stoploop 18 solve-then-loop pairs. Do NOT re-add dpo_synth (already inside).
rows = [json.loads(l) for l in open("dpo_v5_runb.jsonl")]
ds = Dataset.from_list(rows)
print(len(ds), "pairs — expect 51")
assert all(r["chosen"] != r["rejected"] and len(r["rejected"]) >= 2 for r in ds)
import collections
print(collections.Counter(r["meta"].get("rejected_stop") for r in ds))
t = tokenizer.apply_chat_template(ds[0]["prompt"] + ds[0]["chosen"], tools=None, tokenize=False)
assert "<tool_call>" in t and "<tool_response>" in t, "template dropped tool turns"
print(t[-1200:])

#Train (DPO)

In [ ]:
from trl import DPOTrainer, DPOConfig
trainer = DPOTrainer(model=model, ref_model=ref_model, args=DPOConfig(
    beta=0.1, max_length=8192, max_prompt_length=2048,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    # 69 pairs = ~8 optimizer steps: warmup must be tiny, and lr halved vs v1/v2 (margins exploded)
    num_train_epochs=1, learning_rate=2e-5, warmup_steps=1, logging_steps=1,
    optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
    seed=3407, report_to="none",
    output_dir="/content/drive/MyDrive/re_dpo_v5b_ckpt", save_strategy="epoch", save_total_limit=2),
    train_dataset=ds, processing_class=tokenizer)
trainer.train()

Save Adapter

In [ ]:
model.save_pretrained("/content/drive/MyDrive/re_lora_dpo_v5b")
tokenizer.save_pretrained("/content/drive/MyDrive/re_lora_dpo_v5b")

Export GGUF to drive

In [ ]:
model.save_pretrained_gguf("/content/re_dpo_v5b_gguf", tokenizer, quantization_method="q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

... (3 outputs left out)

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [06:51<00:00, 205.58s/it]


Unsloth: Merge process complete. Saved to `/content/re_dpo_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12572-mix-8eebad4 (app-b12572-mix-8eebad4-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/re_dpo_gguf_gguf/Qwen3-4B-Instruct-2507.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions comple

{'save_directory': '/content/re_dpo_gguf',
 'gguf_directory': '/content/re_dpo_gguf_gguf',
 'gguf_files': ['/content/re_dpo_gguf_gguf/Qwen3-4B-Instruct-2507.Q4_K_M.gguf'],
 'modelfile_location': '/content/re_dpo_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

pull the .gguf back into the drive (optional)

In [ ]:
import glob, shutil
# Try both directory options
g_candidates = glob.glob("/content/re_dpo_v5b_gguf_gguf/*.Q4_K_M.gguf") + glob.glob("/content/re_dpo_v5b_gguf/*.Q4_K_M.gguf")
if g_candidates:
    g = g_candidates[0]
    shutil.copy(g, "/content/drive/MyDrive/re-dpo-v5b.gguf")
    print(f"Copied: {g}")
else:
    print("Error: No GGUF files found to copy.")

Copied: /content/re_dpo_gguf_gguf/Qwen3-4B-Instruct-2507.Q4_K_M.gguf


In [ ]:
import glob
import os
from google.colab import files

# Search across both potential directory naming conventions
files_found = (
    glob.glob("/content/re_dpo_v5b_gguf_gguf/*.Q4_K_M.gguf") +
    glob.glob("/content/re_dpo_v5b_gguf_gguf/*.q4_k_m.gguf") +
    glob.glob("/content/re_dpo_v5b_gguf/*.Q4_K_M.gguf") +
    glob.glob("/content/re_dpo_v5b_gguf/*.q4_k_m.gguf")
)

if not files_found:
    print("Error: No Q4_K_M GGUF files found.")
    for d in ["/content/re_dpo_v5b_gguf_gguf", "/content/re_dpo_v5b_gguf"]:
        if os.path.exists(d):
            print(f"Directory contents of {d}:", os.listdir(d))
else:
    print(f"Downloading: {files_found[0]}")
    files.download(files_found[0])

Downloading: /content/re_dpo_gguf_gguf/Qwen3-4B-Instruct-2507.Q4_K_M.gguf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>